# TM Forum RAG — Quickstart Notebook

**Open Telco RAG**

This notebook walks through the Open Telco RAG pipeline for the
**TM Forum** corpus — GitHub repositories covering TMF Open APIs and ODA.

By the end of this notebook you will have:
- Ingested TM Forum API repositories into a local ChromaDB
- Searched TMF Open API specifications
- Asked questions about intent management (TMF921) and ODA
- Evaluated retrieval quality with the evaluation metrics

---

### Prerequisites

```bash
git clone https://github.com/gsma-open-telco-ai/open-telco-rag
cd open-telco-rag
pip install -e .
```

Clone TM Forum repositories:
```bash
python scripts/clone_tmforum.py --output data/tmforum
```


Set your API keys in a `.env` file:
```
# For answer generation (choose one or both)
ANTHROPIC_API_KEY=sk-ant-...        # Claude models direct
OPENRO UTER_API_KEY=sk-or-...        # Any model via OpenRouter
```

## 1. Setup

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv
from collections import Counter
import statistics

load_dotenv()

DATA_DIR   = Path("data/tmforum")
DB_PATH    = "chroma_db"
MODEL_KEY  = "minilm"
COLLECTION = f"tmforum_{MODEL_KEY}"

print(f"Data directory : {DATA_DIR}")
print(f"ChromaDB path  : {DB_PATH}")
print(f"Embedding model: {MODEL_KEY}")
print(f"Collection     : {COLLECTION}")

## 2. Check available content

The TM Forum corpus covers TMF Open API repositories.
Key content for network AI: TMF921 (Intent API),
TMF915 (AI Management Suite), TMF931 (Open Gateway).

In [ ]:
if DATA_DIR.exists():
    dirs  = [d for d in DATA_DIR.iterdir() if d.is_dir()]
    print(f"Repositories : {len(dirs)}")
    print("\nKey API repos:")
    key_repos = ["TMF921_Intent", "TMF915_AIManagementSuite", "TMF931_OpenGateway",
                 "TMF641_ServiceOrder", "TMF620_ProductCatalog", "TMForum-ODA-Ready-for-publication"]
    for r in key_repos:
        path = DATA_DIR / r
        status = "✓ present" if path.exists() else "✗ not found"
        print(f"  {r:<50} {status}")
else:
    print(f"[INFO] {DATA_DIR} not found.")
    print("Clone repos: python scripts/clone_tmforum.py --output data/tmforum")


## 3. Ingest into ChromaDB

The TM Forum ingest pipeline:
1. Loads API YAML and Markdown documents
2. Applies paragraph-based chunking with overlap for DOCX/PDF
3. Excludes tooling repos (alpaca.cpp, text-generation-webui, openapi-generator, etc.)
4. Extracts metadata: repo type, TMF number, document version
5. Embeds and stores with TMF API identifier in metadata

See Section 8 for the chunk size limitation discussion.

In [ ]:
from open_telco_rag.sources.tmforum import TMForumSource
from open_telco_rag.embedders.sentense_transformers import SentenceTransformerEmbedder
from open_telco_rag.stores.chroma import ChromaStore
from open_telco_rag.core.pipeline import ingest

source   = TMForumSource()
embedder = SentenceTransformerEmbedder(model_key=MODEL_KEY)
store    = ChromaStore(path=DB_PATH, collection_name=COLLECTION)

print(f"Starting ingest — {store.count()} chunks already in collection")

if DATA_DIR.exists():
    stats = ingest(
        source     = source,
        store      = store,
        embedder   = embedder,
        data_dir   = str(DATA_DIR),
        chunk_size = 1024,
        overlap    = 200,
        resume     = True,
    )
    print(f"\nIngest complete")
    print(f"  Files processed : {stats.files_processed}")
    print(f"  Chunks stored   : {stats.chunks_stored}")
    print(f"  Skipped         : {stats.skipped_files}")
    print(f"  Errors          : {stats.error_files}")
print(f"  Total in DB     : {store.count():,}")

## 4. Explore the Collection

In [ ]:
import chromadb

client     = chromadb.PersistentClient(path=DB_PATH)
collection = client.get_collection(COLLECTION)
sample     = collection.get(limit=2000, include=["documents", "metadatas"])

tmf_counts  = Counter(m.get("tmf_number", m.get("ig_number", "other")) for m in sample["metadatas"])
type_counts = Counter(m.get("repo_type", "?") for m in sample["metadatas"])
lengths     = [len(d) for d in sample["documents"]]

print(f"Collection: {COLLECTION}")
print(f"Total chunks: {collection.count():,}\n")
print(f"Repository types (sample of 2,000):")
for t, count in type_counts.most_common():
    print(f"  {t:<30} {count}")

print(f"\nTop documents by chunk count:")
print(f"{'Document':<30} {'Chunks':>8}")
print("-" * 40)
for doc, count in tmf_counts.most_common(12):
    print(f"{doc:<30} {count:>8}")

print(f"\nChunk length statistics (chars):")
print(f"  Mean   : {statistics.mean(lengths):.0f}")
print(f"  Median : {statistics.median(lengths):.0f}")

## 5. Semantic Search

TM Forum Open APIs cover intent management, service ordering, and ODA.
Searching across these APIs demonstrates multi-document retrieval.

In [ ]:
from open_telco_rag.core.pipeline import search

QUERY = "What does the TM Forum intent API TMF921 define?"
TOP_K = 5

results = search(
    QUERY,
    store    = store,
    embedder = embedder,
    top_k    = TOP_K,
)

print(f"Query: '{QUERY}'")
print(f"Top-{TOP_K} results:\n")
print(f"{'Rank':<5} {'Similarity':>10} {'Source (truncated)'}")
print("-" * 70)

for i, r in enumerate(results, 1):
    sim = round(1 - r.distance, 3)
    src = r.metadata.get("source", "")[:55]
    print(f"{i:<5} {sim:>10.3f} {src}")

print(f"\nTop result preview:")
print(results[0].text[:400])

### 5.1 Effect of top-k on retrieval

Increasing top-k retrieves more chunks but can also pull in tangential
documentation. Try different values and observe the diversity of sources.

In [ ]:
QUERY2 = "How does TMF921 define the intent lifecycle management?"

print(f"Query: '{QUERY2}'\n")
print(f"{'top-k':<8} {'Unique docs':<14} {'Mean sim':>10} {'Sources'}")
print("-" * 75)

for k in [3, 5, 8, 10]:
    results = search(QUERY2, store=store, embedder=embedder, top_k=k)
    sims    = [1 - r.distance for r in results]
    docs    = list({r.metadata.get('source', '?').split('/')[0] for r in results})
    mean_s  = sum(sims) / len(sims) if sims else 0
    print(f"{k:<8} {len(docs):<14} {mean_s:>10.3f} {', '.join(sorted(docs)[:3])}")

## 6. Question Answering

In [ ]:
from open_telco_rag.core.pipeline import generate

QUESTION = "What does the TM Forum intent API TMF921 define?"

retrieved = search(QUESTION, store=store, embedder=embedder, top_k=5)

from open_telco_rag.generators.openrouter import OpenRouterGenerator
generator = OpenRouterGenerator(model="meta-llama/llama-3.3-70b-instruct")

response  = generate(QUESTION, retrieved, source_type="tmforum", generator=generator)

print(f"Question: {QUESTION}")
print(f"{'─'*70}")
print(f"\n{response['answer']}")
print(f"\n{'─'*70}")
print(f"Sources used: {response['sources']}")
print(f"Chunks used : {response['chunks_used']}")

### 6.1 Question bank

Try the full TM Forum evaluation question set covering
TMF Open APIs and autonomous network architecture.

In [ ]:
QUESTIONS = [
    ("How does TMF641 Service Order API work?",                                       "TMF641"),
    ("What is the TM Forum Open Digital Architecture?",                               "TMForum-ODA"),
    ("What does the TM Forum intent API TMF921 define?",                              "TMF921"),
    ("How does the TMF640 Service Activation API work?",                              "TMF640"),
    ("What is the TM Forum AI Management Suite TMF915?",                              "TMF915"),
    ("How does the TMF Open API programme relate to CAMARA?",                         "TMF931"),
    ("What does the TMF Product Catalog API TMF620 provide?",                         "TMF620"),
]

print(f"{'#':<3} {'Expected doc':<15} {'Top retrieved':<20} {'Mean sim':>9} {'Question'}")
print("-" * 95)

for i, (question, expected) in enumerate(QUESTIONS, 1):
    results  = search(question, store=store, embedder=embedder, top_k=5)
    sims     = [1 - r.distance for r in results]
    mean_sim = sum(sims) / len(sims) if sims else 0
    top_src  = results[0].metadata.get('source', '').split('/')[0][:18] if results else '?'
    hit      = "✓" if expected.lower() in top_src.lower() else "✗"
    q_short  = question[:42] + "..." if len(question) > 42 else question
    print(f"{i:<3} {expected:<15} {top_src + ' ' + hit:<20} {mean_sim:>9.3f} {q_short}")

## 7. Evaluation

Run the evaluation metrics across all TM Forum evaluation questions.

**Note**: metric scoring calls the LLM API (configured below).
The full run takes approximately 3–5 minutes for 10 questions.

In [ ]:
from open_telco_rag.evaluation.evaluate import evaluate_corpus
from open_telco_rag.evaluation.questions import load_questions

questions = load_questions("tmforum")
print(f"Loaded {len(questions)} evaluation questions for TM Forum")
print(f"Questions with reference answers: {sum(1 for q in questions if q.reference_answer)}")

In [ ]:
from open_telco_rag.evaluation._claude import make_llm_call

# RAG generation model (answers questions)
rag_llm_call = make_llm_call("openrouter", "meta-llama/llama-3.3-70b-instruct")

# Metrics evaluation model (scores faithfulness, relevancy, etc.)
llm_call = make_llm_call("openrouter", "anthropic/claude-sonnet-4.5")

result = evaluate_corpus(
    corpus       = "tmforum",
    store        = store,
    embedder     = embedder,
    questions    = questions,
    model        = MODEL_KEY,
    top_k        = 5,
    delay        = 1.0,
    llm_call     = llm_call,
    rag_llm_call = rag_llm_call,
)


In [ ]:
print(f"\nPer-question results — TM Forum ({MODEL_KEY}, top-k=5)")
print(f"{'#':<3} {'Faith':>6} {'Relev':>6} {'Sim':>6} {'Recall':>7} {'Correct':>8} {'Question'}")
print("-" * 95)

for i, qr in enumerate(result.question_results, 1):
    q_short = qr.question[:45] + "..." if len(qr.question) > 45 else qr.question
    print(
        f"{i:<3} "
        f"{qr.faithfulness:>6.2f} "
        f"{qr.answer_relevancy:>6.2f} "
        f"{qr.mean_similarity:>6.3f} "
        f"{max(0, qr.context_recall):>7.2f} "
        f"{max(0, qr.answer_correctness):>8.2f} "
        f"{q_short}"
    )

print("-" * 95)
print(
    f"{'AVG':<3} "
    f"{result.mean_faithfulness:>6.2f} "
    f"{result.mean_answer_relevancy:>6.2f} "
    f"{result.mean_similarity:>6.3f} "
    f"{max(0, result.mean_context_recall):>7.2f} "
    f"{max(0, result.mean_answer_correctness):>8.2f}"
)


## 8. What the metrics tell us

| Metric | Type | Interpretation |
|--------|------|----------------|
| **Faithfulness** | Generation | Are the claims in the answer grounded in the retrieved chunks? |
| **Answer relevancy** | Generation | Does the answer address what was asked? |
| **Mean similarity** | Retrieval | How confident is the retrieval? Higher = better chunk matches |
| **Context recall** | Retrieval+GT | Were the chunks needed to answer the question retrieved? |
| **Answer correctness** | Generation+GT | Is the answer factually correct vs the reference answer? |
